# N1 - E01's placement method as a baseline (highest priority)

**Question.** Kaliuzhnyi et al. (EPSR 265, 2027) maximise the *minimum pairwise Mahalanobis
distance* between fault scenarios and stop when the implied Bayes error drops below
`eps = 1e-3`. What does that criterion achieve on **our** problem — a voltage-only snapshot
at a handful of buses?

**Why it matters.** E01 is the closest prior art, and `novelty_risk.md` says the single
weakest point of our contribution is the claim that our placement objective is different.
Either E01's criterion beats DMP (their approach is stronger and we must say so), or it
does not, and we can show *why* on our own data.

**What is implemented.** Eq. (25) objective, Eq. (19) Bayes link, Eq. (27) stopping rule,
greedy sequential addition — each cited in `e01.py`. Two variants are reported:

| variant | what it minimises over |
|---|---|
| `strict` | every class-different scenario pair — the paper's literal Eq. (25) |
| `visible` | only pairs where at least one event is visible above the noise floor |

The `visible` variant exists because the strict objective **collapses to zero on our data**
(see the diagnostic cell). It is a diagnostic, not a claim about E01.

In [ ]:
# ============================================================================
#  SETUP - run this cell first. Change SMOKE and, if you like, N_JOBS.
# ============================================================================
SMOKE = False         # True: ~1 feeder, 1 seed, small N, minutes. False: full run.

# Optional: restrict this execution to a subset of work units. Leave empty for everything.
# Set e.g. ONLY_FEEDERS = [123] to run just the largest feeder, or ONLY_KS = [2, 3] to run a
# slice of the k grid. Every notebook is resumable, so a sliced run can be continued later
# without redoing finished units (state lives in /kaggle/working/results).
ONLY_FEEDERS = []
ONLY_KS = []

import os, sys, json, time, platform, warnings
warnings.filterwarnings("ignore")
t_start = time.time()

# Where the attached Kaggle dataset is mounted. Everything we need (code, .npz data and
# the precomputed design grids) lives under this one directory.
import glob as _g
_c = sorted(_g.glob("/kaggle/input/**/experiments/k4lib.py", recursive=True))
DATA = os.path.dirname(_c[0]) if _c else "/kaggle/input/snapshot-fd-code-data/experiments"
print("DATA =", DATA)
if not os.path.isdir(DATA):
    # local fallback so the notebook also runs on a laptop for a smoke test
    DATA = os.environ.get("K4_DATA", DATA)
assert os.path.isdir(DATA), (
    "Attach the dataset 'snapshot-fd-code-data' (Add input -> Your datasets). "
    "Expected /kaggle/input/snapshot-fd-code-data/experiments")
# The dataset root also carries k4lib.py and e01.py next to experiments/.
# On Kaggle that is /kaggle/input/snapshot-fd-code-data; locally the task folder.
ROOT = os.environ.get("K4_ROOT", os.path.dirname(DATA))
for p in (DATA, ROOT):
    if p and p not in sys.path:
        sys.path.insert(0, p)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath(".")
RESULTS = os.path.join(WORK, "results")
os.makedirs(RESULTS, exist_ok=True)
os.environ["K4_RESULTS"] = RESULTS          # k4lib reads this
os.environ["K4_DATA"] = DATA

import numpy as np

# OpenDSS is not needed (design grids are precomputed); placement.py imports simlib,
# which imports opendssdirect at load time. Provide an empty stand-in if it is missing.
try:
    import opendssdirect  # noqa: F401
except ImportError:
    import types as _t
    sys.modules["opendssdirect"] = _t.ModuleType("opendssdirect")

import k4lib as K
import features as F
import placement as P
import placement2 as Q

# forward the optional work-slicing from the SETUP cell into the library
K.ONLY_FEEDERS = list(ONLY_FEEDERS)
K.ONLY_KS = list(ONLY_KS)

print("=== environment ===")
K.print_env()
print("SMOKE =", SMOKE)
if ONLY_FEEDERS:
    print("ONLY_FEEDERS =", ONLY_FEEDERS)
if ONLY_KS:
    print("ONLY_KS =", ONLY_KS)
print("elapsed", round(time.time() - t_start, 1), "s")

## 1. How many design-grid events are physically invisible?

Before comparing methods, establish the fact that drives everything below: a fault whose
voltage dip is below the noise floor at every bus cannot be separated by any sensor set.

In [ ]:
import e01
from scipy.stats import norm

feeders = K.feeder_list(SMOKE)
NOISE_LEVELS = K.NOISE_MAIN            # ["1% TVE", "uPMU"], corrected P99 definition
DRAWS = 8 if SMOKE else 40
EPS = 1e-3                             # E01 p.12 adopts eps = 1e-3 for classification

print("noise model (s_m, s_a_deg) and its TVE statistics:")
for name, (ms, asg) in K.NOISE.items():
    q = K.tve_quantiles(ms, asg)
    print(f"  {name:18s} s_m={ms*100:.4f}%  s_a={asg:.4f} deg  "
          f"TVE median {q['median']:.3f}%  P99 {q['p99']:.3f}%  P(>1%) {q['frac_above_1pct']:.3f}")

visibility = {}
for nb in feeders:
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    B = len(buses)
    Mp = np.full((len(Vp), B, 3), np.nan + 0j); Mf = np.full((len(Vp), B, 3), np.nan + 0j)
    v = cols >= 0
    Mp[:, v], Mf[:, v] = Vp[:, cols[v]], Vf[:, cols[v]]
    with np.errstate(all="ignore"):
        dip = np.nan_to_num(1 - np.abs(Mf) / np.abs(Mp), nan=0.0)
    worst = dip.max(axis=(1, 2))
    visibility[nb] = worst
    print(f"IEEE-{nb}: {len(Vp)} simulations")
    for noise in NOISE_LEVELS:
        fl = e01.noise_floor_pu(K.NOISE[noise][0])
        n = int((worst < fl).sum())
        print(f"   {noise:8s}: floor 3*s_m = {fl:.5f} pu -> {n} sims ({100*n/len(Vp):.1f}%) invisible everywhere")


## 2. Place with E01's criterion, both variants

Greedy sequential addition (E01 p.6-7), first PMU fixed at the substation (E01 p.7).

In [ ]:
placement_rows = []
e01_cache = {}
pair_info = {}

for nb in feeders:
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    cand = list(range(len(buses)))
    sub = buses.index(K.dataset(nb).substation) if K.dataset(nb).substation in buses else 0
    kmax = K.ks_max(nb, SMOKE)
    for noise in NOISE_LEVELS:
        ms, asg = K.NOISE[noise]
        reps = {}
        t = time.time()
        for variant, (rep, mode, rfmax) in e01.E01_VARIANTS.items():
            if rep not in reps:
                f = e01._whitened_signatures if rep == "signature" else e01.phasor_signatures
                reps[rep] = f(cols, Vp, Vf, ms, asg, draws=DRAWS, seed=0)
            mu, sd = reps[rep]
            pairs, info = e01.scenario_pairs_v2(meta, mode=mode, rf_max=rfmax,
                                                max_pairs=100000 if SMOKE else 400000)
            pair_info[(nb, noise, variant)] = info
            r = e01.e01_place(mu, sd, pairs, cand, k_max=kmax, eps=EPS, first=sub)
            e01_cache[(nb, noise, variant)] = r
            for rec in r["path"]:
                placement_rows.append(dict(feeder=nb, noise=noise, variant=variant, k=rec["k"],
                                           bus=buses[rec["bus"]], dm=rec["dm"], pe=rec["pe"],
                                           n_pairs=info["n_pairs"], subsampled=info["subsampled"]))
            print(f"IEEE-{nb} {noise:8s} {variant:10s} pairs {info['n_pairs']:7d} "
                  f"{[buses[i] for i in r['buses']]}  D_M {r['path'][-1]['dm']:.2f}  stop_k {r['stop_k']}")
        print(f"   ({time.time()-t:.1f}s)")
print(f"\nD_M required for eps=1e-3: {e01.dm_for_pe(EPS):.3f}")


## 3. Does E01's stopping rule ever fire on our data?

This is the headline diagnostic. E01's own feeder is a 20 kV urban cable network with
`Rf <= 10 ohm` (p.11). Ours spans `Rf` up to 100 ohm with a much weaker voltage-only signal.

In [ ]:
print("=== does Eq. (27) fire? (eps = 1e-3) ===")
for (nb, noise, variant), r in sorted(e01_cache.items()):
    fired = r["stop_k"]
    print(f"IEEE-{nb} {noise:8s} {variant:10s}: "
          f"{'FIRED at k=' + str(fired) if fired else 'never fired'} "
          f"(final D_M {r['path'][-1]['dm']:.2f} vs needed {r['dm_star']:.2f})")


## 4. Compare the resulting sensor sets against DMP and the 11 existing baselines

Same classifiers, same folds, same noise, paired Wilcoxon — so the numbers sit next to the
manuscript's placement table. The eleven baselines (MRA-ILP, QR-pivot, D-optimal, MI-greedy,
FSNR-SVM, GA+LDA, SA, observability, topology, greedy, random) are **loaded** from
`e14_placement_baselines.json` rather than re-run: they were produced by
`e14_placement_baselines.py` with the identical `FAST` model and fold definition, so
recomputing them here would only consume session time. E01 is newly computed on the same
folds, which is what keeps the comparison fair.

In [ ]:
# All placements are evaluated HERE, on the same folds and at the CURRENT noise definition.
# The accuracies stored in e14/e16 were obtained with the legacy noise model and are not
# comparable any more, so only their bus SETS are reused; DMP is also re-designed at the
# evaluation noise level ("DMP @noise").
comp_rows = []

for nb in feeders:
    ds = K.dataset(nb)
    buses, cols, Vp, Vf, meta = K.design_grid(nb)
    gi = K.rows_for_buses(ds, buses)                 # grid bus index -> dataset bus index
    ks = K.k_list(nb, SMOKE)
    sub = np.random.RandomState(0).choice(ds.n, min(1200, ds.n), replace=False) if SMOKE else None
    for noise in NOISE_LEVELS:
        ms, asg = K.NOISE[noise]
        _, marg_n, _ = P.margins(cols, Vp, Vf, meta, mag_sigma=ms, ang_sigma_deg=asg,
                                 draws=DRAWS)
        for k in ks:
            sets = {}
            for name, v in K.baselines(nb, k).items():
                if isinstance(v, dict) and v.get("buses"):
                    sets[name + " (recorded set)"] = ds.bus_ids([str(b) for b in v["buses"]])
            sets["DMP @noise"] = [gi[i] for i in P.dmp(marg_n, k)]
            for variant in e01.E01_VARIANTS:
                r = e01_cache.get((nb, noise, variant))
                if r and len(r["buses"]) >= k:
                    sets[f"E01-{variant}"] = [gi[i] for i in sorted(r["buses"][:k])]
            for name, obs in sets.items():
                X = K.features_of(ds, obs, 0, noise, "FPS")
                if sub is not None:
                    res = K.folds_acc(X[sub], ds.y[sub], ds.rf[sub], seeds=(0,))
                else:
                    res = K.folds_acc(X, ds.y, ds.rf, seeds=(0, 1))
                comp_rows.append(dict(feeder=nb, k=k, noise=noise, method=name,
                                      buses=[ds.buses[i] for i in obs],
                                      acc=res["acc"], acc_sd=res["acc_sd"], hif=res["hif"]))
                print(f"IEEE-{nb} k={k} {noise:8s} {name:32s} acc {res['acc']*100:5.1f}  "
                      f"hif {res['hif']*100:5.1f}")
print(f"\n{len(comp_rows)} rows")


## 5. Save the placement paths (needed by N4 and N6)

Written as individual units so a 12 h session can be resumed.

In [ ]:
def save_one(key):
    nb, noise, variant = key["feeder"], key["noise"], key["variant"]
    r = e01_cache[(nb, noise, variant)]
    buses, *_ = K.design_grid(nb)
    return dict(feeder=nb, noise=noise, variant=variant,
                buses=[buses[i] for i in r["buses"]],
                dm=[p["dm"] for p in r["path"]], pe=[p["pe"] for p in r["path"]],
                stop_k=r["stop_k"], eps=r["eps"], dm_star=r["dm_star"],
                pairs=pair_info[(nb, noise, variant)], meta=K.meta())

keys = [{"feeder": nb, "noise": nz, "variant": v}
        for nb in feeders for nz in NOISE_LEVELS for v in e01.E01_VARIANTS]
K.run_units("n1v2", keys, save_one)

import pandas as pd
pd.DataFrame(placement_rows).to_csv(os.path.join(WORK, "N1_placement_path.csv"), index=False)
pd.DataFrame(comp_rows).to_csv(os.path.join(WORK, "N1_comparison.csv"), index=False)
print("wrote N1_placement_path.csv, N1_comparison.csv")
print("TOTAL", round((time.time()-t_start)/60, 1), "min")
